In [1]:
import gc
import torch

try: del model, tokenizer, image_processor, m3_generator
except: pass
torch.cuda.empty_cache()
gc.collect()

print(f"GPU memory cleared: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

GPU memory cleared: 0.00 GB


In [2]:
import sys
sys.path.append('/workspace/thirdparty/VILA')

from llava.model.builder import load_pretrained_model
from PIL import Image
import requests

model_path = 'MONAI/Llama3-VILA-M3-8B'

print("Loading VILA-M3 model...")
tokenizer, model, image_processor, context_len = load_pretrained_model(
    model_path=model_path,
    model_base=None,
    model_name='model_vila',
    device='cuda:0',
    device_map='auto',
    load_4bit=True if model_path == 'MONAI/Llama3-VILA-M3-13B' else False,
    torch_dtype=torch.float16 
)
conv_mode = ("llama_3" if model_path == 'MONAI/Llama3-VILA-M3-8B' else "vicuna_v1")
print("Model loaded successfully!")

/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status


Loading VILA-M3 model...


/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Fetching 21 files:   0%|          | 0/21 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Model loaded successfully!


In [3]:
import torch
import os

print(f"number of GPU: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    allocated = torch.cuda.memory_allocated(i) / 1024**3
    reserved = torch.cuda.memory_reserved(i) / 1024**3
    total = props.total_memory / 1024**3
    print(f"GPU {i} ({props.name}): {allocated:.2f}GB / {total:.2f}GB (Reserved: {reserved:.2f}GB)")

print(f"\nCUDA_VISIBLE_DEVICES: {os.environ.get('CUDA_VISIBLE_DEVICES', 'Not set')}")

number of GPU: 2
GPU 0 (NVIDIA RTX A5000): 15.96GB / 23.99GB (Reserved: 15.97GB)
GPU 1 (NVIDIA RTX A5000): 0.00GB / 23.99GB (Reserved: 0.00GB)

CUDA_VISIBLE_DEVICES: Not set


In [4]:
import sys
import torch
import torch.nn.functional as F
from typing import Optional
from PIL import Image
import pickle
from datetime import datetime
from llava.conversation import conv_templates, SeparatorStyle
from llava.mm_utils import process_images, tokenizer_image_token
import subprocess
import os

try:
    import torch.distributed as dist
    if dist.is_available() and not dist.is_initialized():
        def _all_reduce_noop(tensor, op=None, group=None, async_op=False):
            return tensor 
        def _get_world_size_noop(group=None):
            return 1
        dist.all_reduce = _all_reduce_noop
        dist.get_world_size = _get_world_size_noop
except Exception:
    pass

sys.path.append('/root/VLM-Radiology-Agent-Framework/m3/demo')

class SimpleM3Generatorv2:
    def __init__(self, model, tokenizer, image_processor, conv_mode):
        self.model = model
        self.tokenizer = tokenizer
        self.image_processor = image_processor
        self.image_cache = {} 

        if conv_mode not in conv_templates:
            available = ", ".join(conv_templates.keys())
            raise ValueError(f"conv_mode '{conv_mode}' not found. Available: {available}")
        self.conv_mode = conv_mode

    def generate_response_simple(self, image_path, prompt, filepath, max_tokens, temperature, top_p):
        if image_path in self.image_cache:
            images_input = self.image_cache[image_path]
        else:
            image = Image.open(image_path).convert("RGB")
            images_input = process_images([image], self.image_processor, self.model.config)
            images_input = images_input.to(self.model.device, dtype=torch.float16)
            self.image_cache[image_path] = images_input
    
        conv = conv_templates[self.conv_mode].copy()
        conv.append_message(conv.roles[0], f"<image>\n{prompt}")
        conv.append_message(conv.roles[1], None)
        prompt_formatted = conv.get_prompt()

        input_ids = tokenizer_image_token(
            prompt_formatted, self.tokenizer, return_tensors="pt"
        )
        if input_ids.dim() == 1:
            input_ids = input_ids.unsqueeze(0)
        input_ids = input_ids.to(self.model.device, dtype=torch.long)

        attention_mask = torch.ones_like(input_ids, device=self.model.device)

        stop_str = conv.sep if conv.sep_style != SeparatorStyle.TWO else conv.sep2
        eos_id = self.tokenizer.eos_token_id
        eot_id = getattr(self.tokenizer, "eot_token_id", None)

        generated = input_ids.clone()
        output_tokens = []
        past_key_values = None
        step_logits = []  
        
        with torch.inference_mode():
            for _ in range(max_tokens):
                labels_dummy = torch.full_like(generated if past_key_values is None else generated[:, -1:], -100, device=self.model.device)

                if past_key_values is None:
                    outputs = self.model(
                        input_ids=generated,
                        images=images_input,
                        attention_mask=attention_mask,
                        use_cache=True,           
                        labels=labels_dummy,      
                        return_dict=True,         
                        dpo_forward=False,
                    )
                else:
                    last_token = generated[:, -1:]
                    last_mask  = torch.ones_like(last_token, device=self.model.device)

                    outputs = self.model(
                        input_ids=last_token,
                        images=images_input,      
                        attention_mask=last_mask,
                        use_cache=True,
                        past_key_values=past_key_values,   
                        labels=labels_dummy,
                        return_dict=True,
                        dpo_forward=False,
                    )
                
                # ... outputs ...
                last_logits = outputs.logits[:, -1, :].detach().cpu().float()  # (1, V)
                step_logits.append(last_logits)
                
                # base_logits = outputs['logits'].cpu().float()
                # # Save
                # output_dict = {
                #     'logits': base_logits,
                # }
                # with open(filepath, "wb") as f:
                #     pickle.dump(output_dict, f)

                # # -- save outputs
                # with open(filepath, "wb") as f:
                #     pickle.dump(outputs, f)
                
                past_key_values = outputs.past_key_values   

                logits = outputs.logits[:, -1, :]

                # --- decoding ---
                if temperature > 0:
                    probs = F.softmax(logits / temperature, dim=-1)
                    if 0.0 < top_p < 1.0:
                        sorted_probs, sorted_indices = torch.sort(probs, descending=True)
                        cumulative = torch.cumsum(sorted_probs, dim=-1)
                        keep = cumulative <= top_p
                        keep[..., 0] = True
                        filtered_probs = torch.where(keep, sorted_probs, torch.zeros_like(sorted_probs))
                        filtered_probs = filtered_probs / filtered_probs.sum(dim=-1, keepdim=True)

                        next_idx_sorted = torch.multinomial(filtered_probs[0], 1)         
                        next_token = sorted_indices[0, next_idx_sorted]                   
                    else:
                        next_token = torch.multinomial(probs[0], 1)                        
                else:
                    next_token = torch.argmax(logits, dim=-1)                               

                next_token = next_token.to(self.model.device, dtype=torch.long).view(1, 1)

                generated = torch.cat([generated, next_token], dim=1)
                attention_mask = torch.cat([attention_mask, torch.ones_like(next_token, device=self.model.device)], dim=1) # <<< YENİ: maskeyi büyüt (her seferinde baştan oluşturma)
                output_tokens.append(next_token.item())

                # early stop
                tid = next_token.item()
                if (eos_id is not None and tid == eos_id) or (eot_id is not None and tid == eot_id):
                    break

        # outputs_text = self.tokenizer.decode(output_tokens, skip_special_tokens=True).strip()
        # if stop_str and outputs_text.endswith(stop_str):
        #     outputs_text = outputs_text[:-len(stop_str)].strip()
        #print(outputs_text)
        # return outputs_text
        
        logits_seq = torch.stack(step_logits, dim=1)   # (1, T_gen, V)

        os.makedirs(os.path.dirname(filepath), exist_ok=True)
        with open(filepath, "wb") as f:
            pickle.dump({"logits": logits_seq}, f)

        return

model.eval()

m3_generatorv2 = SimpleM3Generatorv2(model, tokenizer, image_processor, conv_mode=conv_mode)
print("M3Generator (with model() decoding) is ready!")


M3Generator (with model() decoding) is ready!


In [5]:
"""
EMBEDDING VISUALIZATION FOR VILA-M3
"""

import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
import umap
from pathlib import Path
from PIL import Image

IMAGE_FOLDER = r"/root/VLM-Radiology-Agent-Framework/images" 
OUTPUT_PATH = "/root/VLM-Radiology-Agent-Framework/images/embedding_spaces/vila_embedding_space.png"
METHOD = "tsne"  # "tsne" or "umap"

model = m3_generatorv2.model
image_processor = m3_generatorv2.image_processor
device = model.device

vision_tower = model.get_vision_tower()

if hasattr(vision_tower, 'vision_model'):
    vision_model = vision_tower.vision_model
elif hasattr(vision_tower, 'vision_tower'):
    vision_model = vision_tower.vision_tower
    if hasattr(vision_model, 'vision_model'):
        vision_model = vision_model.vision_model
else:
    vision_model = vision_tower

vision_model.eval()
print(f"✓ Vision model: {type(vision_model)}\n")

image_folder = Path(IMAGE_FOLDER)
image_paths = sorted(list(image_folder.glob("*.png")) + 
                    list(image_folder.glob("*.jpg")))

if len(image_paths) == 0:
    raise FileNotFoundError("No images!")

image_paths = image_paths#[:30]

embeddings = []
labels = []

try:
    from llava.mm_utils import process_images
except ImportError:
    def process_images(images, processor, config):
        return processor.preprocess(images, return_tensors='pt')['pixel_values']

for i, img_path in enumerate(image_paths, start=1):
    try:
        image = Image.open(img_path).convert('RGB') 
        
        images_input = process_images([image], image_processor, model.config)
        images_input = images_input.to(device, dtype=torch.float16)
        
        with torch.no_grad():
            vision_outputs = vision_model(images_input)
            
            if hasattr(vision_outputs, 'last_hidden_state'):
                embedding = vision_outputs.last_hidden_state[:, 0, :].cpu().float().numpy()
            elif hasattr(vision_outputs, 'pooler_output'):
                embedding = vision_outputs.pooler_output.cpu().float().numpy()
            elif isinstance(vision_outputs, tuple):
                embedding = vision_outputs[0][:, 0, :].cpu().float().numpy()
            else:
                embedding = vision_outputs[:, 0, :].cpu().float().numpy()
        
        embeddings.append(embedding[0] if embedding.shape[0] == 1 else embedding)
        labels.append(f"{img_path}")
        
        if i % 5 == 0:
            print(f"   ✓ {i}/{len(image_paths)} completed...")
    
    except Exception as e:
        print(f"   ✗ Error (image {i}): {e}")
        continue

embeddings = np.array(embeddings)
print(f"\n✓ Toplam {len(embeddings)} embeddings were extracted")
print(f"  Shape: {embeddings.shape}\n")

torch.cuda.empty_cache()

print(f"{METHOD.upper()} for 2D...")

if METHOD.lower() == "tsne":
    reducer = TSNE(
        n_components=2,
        perplexity=min(5, len(embeddings) - 1),
        max_iter=1000, 
        random_state=42,
        metric='cosine'
    )
    embeddings_2d = reducer.fit_transform(embeddings)
    method_name = "t-SNE"
else:
    reducer = umap.UMAP(
        n_neighbors=min(5, len(embeddings) - 1),
        min_dist=0.1,
        n_components=2,
        random_state=42,
        metric='cosine'
    )
    embeddings_2d = reducer.fit_transform(embeddings)
    method_name = "UMAP"

print(f"✓ {method_name} completed\n")

import pandas as pd
df = pd.DataFrame({
    'image_id': range(1, len(embeddings_2d) + 1),
    'image_name': labels,
    'dim1': embeddings_2d[:, 0],
    'dim2': embeddings_2d[:, 1],
    'model': 'VILA-M3',
    'encoder': 'CLIP ViT-L/14',
    'method': method_name
})
df.to_excel(r"/root/VLM-Radiology-Agent-Framework/vila_embeddings_2d.xlsx", index=False)

✓ Vision model: <class 'transformers.models.siglip.modeling_siglip.SiglipVisionTransformer'>

   ✓ 5/3309 completed...
   ✓ 10/3309 completed...
   ✓ 15/3309 completed...
   ✓ 20/3309 completed...
   ✓ 25/3309 completed...
   ✓ 30/3309 completed...
   ✓ 35/3309 completed...
   ✓ 40/3309 completed...
   ✓ 45/3309 completed...
   ✓ 50/3309 completed...
   ✓ 55/3309 completed...
   ✓ 60/3309 completed...
   ✓ 65/3309 completed...
   ✓ 70/3309 completed...
   ✓ 75/3309 completed...
   ✓ 80/3309 completed...
   ✓ 85/3309 completed...
   ✓ 90/3309 completed...
   ✓ 95/3309 completed...
   ✓ 100/3309 completed...
   ✓ 105/3309 completed...
   ✓ 110/3309 completed...
   ✓ 115/3309 completed...
   ✓ 120/3309 completed...
   ✓ 125/3309 completed...
   ✓ 130/3309 completed...
   ✓ 135/3309 completed...
   ✓ 140/3309 completed...
   ✓ 145/3309 completed...
   ✓ 150/3309 completed...
   ✓ 155/3309 completed...
   ✓ 160/3309 completed...
   ✓ 165/3309 completed...
   ✓ 170/3309 completed...
   ✓ 175

In [ ]:
import numpy as np
import os
from pathlib import Path

txt_path = "/root/VLM-Radiology-Agent-Framework/image_list_150.txt"
img_dir = "/root/VLM-Radiology-Agent-Framework/images"

with open(txt_path, "r") as f:
    image_names = [line.strip() for line in f if line.strip()]

all_images = os.listdir(img_dir)

img_files = []
for name in image_names:
    match = next((f for f in all_images if f.startswith(name)), None)
    if match:
        img_files.append(match)
    else:
        print(f"Not found in {name} ")

print(f"{len(img_files)} images were found.")

#img_files = [f for f in os.listdir(img_folder) if f.lower().endswith((".jpg", ".jpeg", ".png"))]

print(model_path + " Results with model():\n")

prompts = {
    1: ("This is a histopathology slide showing tissue under microscope."+
        "Please analyze the cellular morphology, tissue architecture, and identify whether this shows normal or abnormal pathological changes."+
        "Focus on: 1) Cell types present 2) Tissue organization 3) Any signs of malignancy or benign changes"),
            
    2: ("Examine this histopathology slide and explain each item:"+ 
        "1) Tissue diagnosis. Evaluate the tissue architecture, noting any disruptions or patterns."+
        "2) Assess the overall tissue organization for normality or abnormalities. "+
        "3) If malignant: histologic grade (well, moderately, or poorly differentiated) with justification."+ 
        "4) Features relevant to staging: invasion depth, lymphovascular invasion, margins. "+
        "5) Prognostic indicators: mitotic count, necrosis, specific molecular markers if identifiable."+ 
        "6) Identify cell types present whether the slide appears normal, benignly abnormal, or potentially malignant."),
    
    3: ("Perform a systematic analysis of this histopathology slide, quantify the following features, and synthesize these findings into a diagnostic interpretation:"+
        "1) Predominant cell types and their approximate proportions."+
        "2) Nuclear-to-cytoplasmic ratio (low/moderate/high)."+
        "3) Mitotic figures: absent/rare/frequent."+
        "4) Degree of pleomorphism: mild/moderate/severe."+
        "5) Architectural disruption: none/focal/diffuse."+
        "6) Inflammatory infiltrate: type and density."),
    11: (
        "You are viewing a microscopic image of tissue from a histopathology slide. "
        "Describe the cell morphology and how the tissue is structurally organized, and state whether the findings appear normal or pathological. "
        "Address: 1) Which cell types are visible 2) How the tissue is arranged 3) Whether there are indications of malignant or benign change"
    ),
    12: (
        "Examine this microscopic tissue image and provide a basic morphological assessment. "
        "Comment on the cells observed, the way the tissue is structured, and whether the appearance is consistent with a normal or an abnormal pathological process. "
        "Cover the following points: cell types identified, tissue organization, and any evidence suggestive of malignancy or a benign condition."
    ),
    13: (
        "Look at this histopathology image and assess the tissue at the cellular level. "
        "Report on the morphology of the cells and the overall architecture of the tissue, concluding with an opinion on whether the changes seen are normal or pathological. "
        "Please specifically note: a) cell types you can identify, b) the tissue's structural pattern, c) any features pointing toward malignancy or a benign process."
    ),
    21: (
        "Review this histopathology slide and address the following points in turn: "
        "1) What is the tissue diagnosis? Comment on the tissue architecture and any disruption or recognizable pattern. "
        "2) Is the overall tissue organization normal or abnormal? "
        "3) If a malignancy is present, provide the histologic grade (well, moderately, or poorly differentiated) along with your reasoning. "
        "4) Note any staging-relevant features such as depth of invasion, lymphovascular invasion, or margin status. "
        "5) Mention prognostic indicators including mitotic count, necrosis, or any identifiable molecular markers. "
        "6) State the cell types present and classify the slide as normal, benign but abnormal, or potentially malignant."
    ),
    22: (
        "Analyze this histopathology slide in a structured manner, covering each of the following: "
        "1) Provide a tissue diagnosis, describing the architecture and highlighting any disrupted or patterned regions. "
        "2) Determine whether the tissue organization overall is normal or shows abnormalities. "
        "3) Should malignancy be present, specify the histologic grade (well/moderately/poorly differentiated) and justify your assessment. "
        "4) Identify staging-related features, including invasion depth, lymphovascular involvement, and margin status. "
        "5) List prognostic markers such as mitotic activity, necrosis, and any identifiable molecular indicators. "
        "6) Specify the cell types observed and indicate whether the tissue is normal, benign with abnormalities, or suspicious for malignancy."
    ),
    23: (
        "Please provide an intermediate-level diagnostic evaluation of this histopathology slide by responding to the items below: "
        "1) Tissue diagnosis — comment on architectural integrity, identifying disruptions or notable patterns. "
        "2) Normality assessment — is the tissue organization within normal limits or abnormal? "
        "3) In case of malignancy, report the histologic grade (well, moderately, poorly differentiated) with supporting rationale. "
        "4) Staging features — comment on invasion depth, lymphovascular invasion, and margin involvement. "
        "5) Prognosis-related findings — mitotic count, necrosis, and molecular markers if visible. "
        "6) Cell type identification — classify the slide as normal, benign/abnormal, or potentially malignant."
    ),
    31: (
        "Conduct a systematic, quantitative evaluation of this histopathology slide and combine your findings into a final diagnostic interpretation. "
        "Specifically report: 1) The dominant cell types and their relative proportions. "
        "2) The nuclear-to-cytoplasmic ratio, categorized as low, moderate, or high. "
        "3) Mitotic activity, categorized as absent, rare, or frequent. "
        "4) The extent of pleomorphism, rated as mild, moderate, or severe. "
        "5) The degree of architectural disruption, described as none, focal, or diffuse. "
        "6) The type and density of any inflammatory infiltrate present."
    ),
    32: (
        "Carry out a quantitative, step-by-step assessment of this histopathology slide, and integrate the results into a single diagnostic conclusion. "
        "Address the following: 1) Identify the predominant cell populations and estimate their proportions. "
        "2) Rate the nuclear-to-cytoplasmic ratio as low, moderate, or high. "
        "3) Characterize mitotic figures as absent, rare, or frequent. "
        "4) Grade the degree of pleomorphism as mild, moderate, or severe. "
        "5) Describe architectural disruption as none, focal, or diffuse. "
        "6) Specify the type and density of inflammatory cell infiltration."
    ),
    33: (
        "Perform a detailed, quantitative analysis of this histopathological image and synthesize your observations into a diagnostic summary. "
        "Your analysis should quantify: 1) The main cell types present and their approximate relative abundance. "
        "2) Nuclear-to-cytoplasmic ratio, classified as low, moderate, or high. "
        "3) The frequency of mitotic figures: absent, rare, or frequent. "
        "4) The severity of cellular pleomorphism: mild, moderate, or severe. "
        "5) The pattern of architectural disruption: none, focal, or diffuse. "
        "6) The nature and density of any inflammatory infiltrate observed."
    ),
}
    
for kk in [1,2,3]:#1,2,3,11,12,13,21,22,23,31,32,33
    prompt = prompts[kk]
    
    print(kk, prompt)
    for i, img_name in enumerate(img_files, start=1):
    
        if i in [135,145]: #>= 26 and i <= 100: 35,84,116,124,146, 61,72,97,109,149, 46,47,57,60,100
            save_dir = f"/root/VLM-Radiology-Agent-Framework/q{kk}/pkl_output_{i}"
            Path(save_dir).mkdir(parents=True, exist_ok=True)
            image_path = os.path.join(img_dir, img_name)
            print(image_path[-40:-4])
            for temperature in [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
                print("temperature: ",temperature)
                while True:
                    count = sum(
                        1 for f in os.listdir(save_dir)
                        if f.endswith(".pkl") and f"_temp_{temperature}_" in f
                    )
                    if count >= 30:
                        break
                    else:
                        print("test no: ", count)
                        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
                        filename = f"outputs_img_{image_path[-40:-4]}_temp_{temperature}_{timestamp}.pkl"
                        filepath = os.path.join(save_dir, filename)
                        response = m3_generatorv2.generate_response_simple(
                            image_path=image_path,
                            prompt=prompt,
                            filepath = filepath,
                            max_tokens=512,
                            temperature=temperature,
                            top_p=0.9,
                        )


150 images were found.
MONAI/Llama3-VILA-M3-8B Results with model():

1 This is a histopathology slide showing tissue under microscope.Please analyze the cellular morphology, tissue architecture, and identify whether this shows normal or abnormal pathological changes.Focus on: 1) Cell types present 2) Tissue organization 3) Any signs of malignancy or benign changes
3f2417b7-b3d0-4672-b62e-959f03ffc47d
temperature:  0.0
temperature:  0.1
temperature:  0.2
temperature:  0.3
temperature:  0.4
temperature:  0.5
temperature:  0.6
temperature:  0.7
temperature:  0.8
temperature:  0.9
temperature:  1.0
6d3bc0c0-df26-4fd0-8887-94fa992d432e
temperature:  0.0
temperature:  0.1
temperature:  0.2
temperature:  0.3
temperature:  0.4
temperature:  0.5
temperature:  0.6
temperature:  0.7
temperature:  0.8
temperature:  0.9
temperature:  1.0
2 Examine this histopathology slide and explain each item:1) Tissue diagnosis. Evaluate the tissue architecture, noting any disruptions or patterns.2) Assess the 